# Pascal VOC 2007 Semantic Segmentation

This notebook runs the full pipeline on Google Colab:
1. Setup & dependencies
2. Download Pascal VOC 2007 from Kaggle
3. Dataset exploration
4. Train 4 models: U-Net, DeepLabV3+, DINOv2, SAM2
5. Inference & visualization
6. Evaluation & cross-model comparison
7. Confusion matrix
8. Qualitative visualizations (mosaic, top-3 best/worst, person class)
9. Ablation studies
10. Display all plots & download as zip

## 0. GPU Check

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("WARNING: No GPU detected. Go to Runtime -> Change runtime type -> GPU.")

## 1. Clone Repo & Install Dependencies

In [ ]:
import os

# Clone repo (change to your repo URL if needed)
REPO_DIR = "/content/261-mini2"
if not os.path.exists(REPO_DIR):
    !git clone https://github.com/gracee-chen/261-mini2.git {REPO_DIR}
os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
# Install dependencies
!pip install -q segmentation-models-pytorch timm transformers scipy

# Install SAM2
!pip install -q git+https://github.com/facebookresearch/sam2.git

## 2. Download Pascal VOC 2007 from Kaggle

In [ ]:
# Upload your kaggle.json API key
# Get it from: https://www.kaggle.com/settings -> API -> Create New Token
from google.colab import files

uploaded = files.upload()  # upload kaggle.json here

!mkdir -p ~/.kaggle
!mv kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
print("Kaggle API key configured.")

In [ ]:
!pip install -q kaggle

# Download dataset
DOWNLOAD_DIR = "/content/voc_download"
EXTRACT_DIR  = os.path.join(REPO_DIR, "voc_data")

if not os.path.exists(EXTRACT_DIR):
    !kaggle datasets download -d zaraks/pascal-voc-2007 -p {DOWNLOAD_DIR}
    !unzip -q {DOWNLOAD_DIR}/pascal-voc-2007.zip -d {EXTRACT_DIR}
    !rm -rf {DOWNLOAD_DIR}
    print("Dataset downloaded and extracted.")
else:
    print("Dataset already exists.")

# Show what's inside after extraction
print("\nExtracted contents:")
!ls -la {EXTRACT_DIR}

# Auto-detect VOC_ROOT: find the directory that contains VOCdevkit/VOC2007/JPEGImages
import pathlib

VOC_ROOT = None
for candidate in [
    EXTRACT_DIR,                                                          # VOCdevkit/ directly inside
    os.path.join(EXTRACT_DIR, "VOCtrainval_06-Nov-2007"),                 # standard wrapper
]:
    target = os.path.join(candidate, "VOCdevkit", "VOC2007", "JPEGImages")
    if os.path.isdir(target):
        VOC_ROOT = candidate
        break

# Fallback: search recursively for VOCdevkit
if VOC_ROOT is None:
    for p in pathlib.Path(EXTRACT_DIR).rglob("VOCdevkit"):
        if (p / "VOC2007" / "JPEGImages").is_dir():
            VOC_ROOT = str(p.parent)
            break

assert VOC_ROOT is not None, (
    f"Could not find VOCdevkit/VOC2007/JPEGImages under {EXTRACT_DIR}. "
    f"Check the zip structure with: !find {EXTRACT_DIR} -maxdepth 4 -type d"
)

jpeg_dir = os.path.join(VOC_ROOT, "VOCdevkit", "VOC2007", "JPEGImages")
seg_dir  = os.path.join(VOC_ROOT, "VOCdevkit", "VOC2007", "SegmentationClass")
print(f"\nVOC_ROOT       = {VOC_ROOT}")
print(f"JPEGImages     : {len(os.listdir(jpeg_dir))} files")
print(f"SegmentationClass exists: {os.path.isdir(seg_dir)}")
if os.path.isdir(seg_dir):
    print(f"SegmentationClass: {len(os.listdir(seg_dir))} files")

## 3. Dataset Exploration

In [ ]:
import sys
sys.path.insert(0, REPO_DIR)

import numpy as np
import matplotlib.pyplot as plt
from dataset.voc_dataset import VOC_CLASSES, NUM_CLASSES, get_datasets, mask_to_class_index

train_ds, val_ds = get_datasets(VOC_ROOT, image_size=256)
print(f"Train samples: {len(train_ds)}")
print(f"Val samples  : {len(val_ds)}")
print(f"Classes ({NUM_CLASSES}): {VOC_CLASSES}")

In [ ]:
import random
import torch

# Visualize random samples
mean = torch.tensor([0.485, 0.456, 0.406]).view(3, 1, 1)
std  = torch.tensor([0.229, 0.224, 0.225]).view(3, 1, 1)

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
indices = random.sample(range(len(train_ds)), 4)

for col, idx in enumerate(indices):
    img, mask = train_ds[idx]
    img_np = (img * std + mean).clamp(0, 1).permute(1, 2, 0).numpy()
    mask_np = mask_to_class_index(mask).numpy()
    mask_display = mask_np.copy()
    mask_display[mask_display == 255] = 0

    axes[0, col].imshow(img_np)
    axes[0, col].set_title(f"Image #{idx}")
    axes[0, col].axis("off")

    seg = axes[1, col].imshow(mask_display, cmap="tab20", vmin=0, vmax=20)
    classes_present = [VOC_CLASSES[c] for c in np.unique(mask_np) if c < 21]
    axes[1, col].set_title(f"{', '.join(classes_present)}")
    axes[1, col].axis("off")

plt.suptitle("Training Samples: Image + Segmentation Mask", fontsize=14)
plt.tight_layout()
plt.show()

## 4. Train Models

Each model is trained via the CLI scripts. Checkpoints are saved to `checkpoints/<model>/`.

### 4.1 U-Net (ResNet-50)

In [ ]:
!python train/train_unet.py \
    --voc-root {VOC_ROOT} \
    --epochs 50 \
    --batch-size 8 \
    --lr 1e-4 \
    --image-size 256

### 4.2 DeepLabV3+ (ResNet-50)

In [ ]:
!python train/train_deeplabv3plus.py \
    --voc-root {VOC_ROOT} \
    --epochs 50 \
    --batch-size 8 \
    --lr 1e-4 \
    --image-size 256

### 4.3 DINOv2 (ViT-B/14)

Two-phase training:
- Phase 1: Freeze backbone, train decoder head only
- Phase 2: Unfreeze backbone, fine-tune end-to-end with lower LR

In [ ]:
# Phase 1: Train head only (backbone frozen)
!python train/train_dinov2.py \
    --voc-root {VOC_ROOT} \
    --epochs 30 \
    --batch-size 16 \
    --lr 1e-3

In [ ]:
# Phase 2: Fine-tune entire model
!python train/train_dinov2.py \
    --voc-root {VOC_ROOT} \
    --unfreeze-backbone \
    --resume checkpoints/dinov2/best.pth \
    --epochs 20 \
    --batch-size 8 \
    --lr 1e-5

### 4.4 SAM2 (Hiera Base+)

Requires downloading the SAM2 checkpoint first.

In [ ]:
# Download SAM2 checkpoint
SAM2_CKPT = os.path.join(REPO_DIR, "sam2.1_hiera_base_plus.pt")
if not os.path.exists(SAM2_CKPT):
    !wget -q https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_base_plus.pt -O {SAM2_CKPT}
    print(f"Downloaded SAM2 checkpoint: {SAM2_CKPT}")
else:
    print("SAM2 checkpoint already exists.")

In [ ]:
!python train/train_sam2.py \
    --voc-root {VOC_ROOT} \
    --sam2-ckpt {SAM2_CKPT} \
    --epochs 30 \
    --batch-size 8 \
    --lr 3e-4

## 5. Inference & Visualization

In [ ]:
# Run inference for each model on VOC val samples
models_to_infer = [
    ("unet",          f"python inference/infer_unet.py --checkpoint checkpoints/unet/best.pth --voc-root {VOC_ROOT} --num-samples 4 --output-dir results/infer_unet"),
    ("deeplabv3plus", f"python inference/infer_deeplabv3plus.py --checkpoint checkpoints/deeplabv3plus/best.pth --voc-root {VOC_ROOT} --num-samples 4 --output-dir results/infer_deeplabv3plus"),
    ("dinov2",        f"python inference/infer_dinov2.py --checkpoint checkpoints/dinov2/best.pth --voc-root {VOC_ROOT} --num-samples 4 --output-dir results/infer_dinov2"),
    ("sam2",          f"python inference/infer_sam2.py --checkpoint checkpoints/sam2/best.pth --sam2-ckpt {SAM2_CKPT} --voc-root {VOC_ROOT} --num-samples 4 --output-dir results/infer_sam2"),
]

for name, cmd in models_to_infer:
    ckpt_path = f"checkpoints/{name}/best.pth"
    if os.path.exists(ckpt_path):
        print(f"\n{'='*60}")
        print(f"  Running inference: {name}")
        print(f"{'='*60}")
        os.system(cmd)
    else:
        print(f"Skipping {name} — checkpoint not found at {ckpt_path}")

In [ ]:
# Display saved inference results
from IPython.display import display, Image as IPImage
import glob

for name in ["unet", "deeplabv3plus", "dinov2", "sam2"]:
    infer_dir = f"results/infer_{name}"
    pngs = sorted(glob.glob(os.path.join(infer_dir, "*.png")))
    if pngs:
        print(f"\n--- {name.upper()} ---")
        for p in pngs[:2]:
            display(IPImage(filename=p, width=700))

## 6. Evaluation

### 6.1 Compute Metrics (mIoU, mDice, HD95, Pixel Accuracy, Per-class IoU/Acc)

In [ ]:
# Build the evaluation command dynamically based on available checkpoints
available_models = []
available_ckpts  = []

for name in ["unet", "deeplabv3plus", "dinov2", "sam2"]:
    ckpt = f"checkpoints/{name}/best.pth"
    if os.path.exists(ckpt):
        available_models.append(name)
        available_ckpts.append(ckpt)

if available_models:
    model_types = " ".join(available_models)
    ckpt_paths  = " ".join(available_ckpts)
    sam2_flag   = f"--sam2-ckpt {SAM2_CKPT}" if "sam2" in available_models else ""

    cmd = (
        f"python evaluation/metrics/compute_metrics.py "
        f"--model-type {model_types} "
        f"--checkpoint {ckpt_paths} "
        f"--voc-root {VOC_ROOT} "
        f"--output-dir results/metrics "
        f"{sam2_flag}"
    )
    print(f"Running: {cmd}\n")
    os.system(cmd)
else:
    print("No checkpoints found. Train models first.")

### 6.2 Cross-Model Comparison (performance, training time, generalization)

In [ ]:
if available_models:
    model_names    = " ".join(available_models)
    ckpt_dirs      = " ".join([f"checkpoints/{m}" for m in available_models])

    cmd = (
        f"python evaluation/compare.py "
        f"--models {model_names} "
        f"--checkpoint-dirs {ckpt_dirs} "
        f"--metrics-dir results/metrics "
        f"--output-dir results/compare"
    )
    print(f"Running: {cmd}\n")
    os.system(cmd)
else:
    print("No models available for comparison.")

In [ ]:
# Display comparison plots
from IPython.display import display, Image as IPImage

compare_dir = "results/compare"
for fname in ["loss_curves.png", "metrics_bar.png", "training_time_bar.png", "generalization_bar.png"]:
    fpath = os.path.join(compare_dir, fname)
    if os.path.exists(fpath):
        print(f"\n--- {fname} ---")
        display(IPImage(filename=fpath, width=800))

# Print summary table
table_path = os.path.join(compare_dir, "summary_table.txt")
if os.path.exists(table_path):
    print("\n" + open(table_path).read())

## 7. Confusion Matrix

Pixel-level confusion between classes for each model.

In [ ]:
# Generate confusion matrix for each available model
for name in available_models:
    ckpt = f"checkpoints/{name}/best.pth"
    sam2_flag = f"--sam2-ckpt {SAM2_CKPT}" if name == "sam2" else ""
    cmd = (
        f"python evaluation/metrics/confusion_matrix.py "
        f"--model-type {name} "
        f"--checkpoint {ckpt} "
        f"--voc-root {VOC_ROOT} "
        f"--output-dir results/metrics "
        f"{sam2_flag}"
    )
    print(f"\n{'='*60}")
    print(f"  Confusion Matrix: {name}")
    print(f"{'='*60}")
    os.system(cmd)

In [ ]:
# Display confusion matrices
from IPython.display import display, Image as IPImage

for name in available_models:
    fpath = f"results/metrics/{name}_confusion_matrix.png"
    if os.path.exists(fpath):
        print(f"\n--- {name.upper()} Confusion Matrix ---")
        display(IPImage(filename=fpath, width=700))

## 8. Qualitative Visualizations

### 8.1 Mosaic: Side-by-side Ground Truth vs Predictions (all models)

In [ ]:
# Build mosaic: Input | Ground Truth | Model1 | Model2 | ...
model_specs = []
for name in available_models:
    ckpt = f"checkpoints/{name}/best.pth"
    model_specs.append(f"{name}:{ckpt}")

if model_specs:
    specs_str = " ".join(model_specs)
    sam2_flag = f"--sam2-ckpt {SAM2_CKPT}" if "sam2" in available_models else ""
    cmd = (
        f"python evaluation/visualization/visualize_mosaic.py "
        f"--voc-root {VOC_ROOT} "
        f"--models {specs_str} "
        f"--num-images 6 "
        f"--output results/visualization/mosaic.png "
        f"{sam2_flag}"
    )
    print(f"Running: {cmd}\n")
    os.system(cmd)

In [ ]:
from IPython.display import display, Image as IPImage

mosaic_path = "results/visualization/mosaic.png"
if os.path.exists(mosaic_path):
    display(IPImage(filename=mosaic_path, width=900))

### 8.2 Top-3 Best & Worst Results (by overall mIoU)

In [ ]:
# Top-3 best and worst results ranked by overall mIoU
if available_models:
    model_types = " ".join(available_models)
    ckpt_paths  = " ".join([f"checkpoints/{m}/best.pth" for m in available_models])
    sam2_flag   = f"--sam2-ckpt {SAM2_CKPT}" if "sam2" in available_models else ""

    cmd = (
        f"python evaluation/visualization/visualize_comparison.py "
        f"--voc-root {VOC_ROOT} "
        f"--model-type {model_types} "
        f"--checkpoint {ckpt_paths} "
        f"--metric miou "
        f"--output-dir results/visualization "
        f"{sam2_flag}"
    )
    print(f"Running: {cmd}\n")
    os.system(cmd)

In [ ]:
from IPython.display import display, Image as IPImage

for fname, label in [("top3_miou.png", "Top-3 Best (mIoU)"), ("worst3_miou.png", "Top-3 Worst (mIoU)")]:
    fpath = os.path.join("results/visualization", fname)
    if os.path.exists(fpath):
        print(f"\n--- {label} ---")
        display(IPImage(filename=fpath, width=800))

### 8.3 Person (Human) Class Analysis

Top-3 best and worst results ranked specifically by Person class IoU.

In [ ]:
# Top-3 best and worst results ranked by Person class IoU
if available_models:
    model_types = " ".join(available_models)
    ckpt_paths  = " ".join([f"checkpoints/{m}/best.pth" for m in available_models])
    sam2_flag   = f"--sam2-ckpt {SAM2_CKPT}" if "sam2" in available_models else ""

    cmd = (
        f"python evaluation/visualization/visualize_comparison.py "
        f"--voc-root {VOC_ROOT} "
        f"--model-type {model_types} "
        f"--checkpoint {ckpt_paths} "
        f"--metric person "
        f"--output-dir results/visualization "
        f"{sam2_flag}"
    )
    print(f"Running: {cmd}\n")
    os.system(cmd)

In [ ]:
from IPython.display import display, Image as IPImage

for fname, label in [("top3_person.png", "Top-3 Best (Person IoU)"), ("worst3_person.png", "Top-3 Worst (Person IoU)")]:
    fpath = os.path.join("results/visualization", fname)
    if os.path.exists(fpath):
        print(f"\n--- {label} ---")
        display(IPImage(filename=fpath, width=800))

## 9. Ablation Studies

Run ablation experiments on U-Net to study the effect of different factors.

In [ ]:
# Backbone ablation: compare different encoder architectures (e.g. ResNet-18 vs ResNet-50)
!python evaluation/ablation/ablation_backbone.py --voc-root {VOC_ROOT}

In [ ]:
# Loss ablation: compare cross-entropy vs Dice loss weight combinations
!python evaluation/ablation/ablation_loss.py --voc-root {VOC_ROOT}

In [ ]:
# Augmentation ablation: train with vs without geometric augmentation
!python evaluation/ablation/ablation_augmentation.py --voc-root {VOC_ROOT}

In [ ]:
# Pretrain ablation: ImageNet pretrained vs training from scratch
!python evaluation/ablation/ablation_pretrain.py --voc-root {VOC_ROOT}

In [ ]:
# Resolution ablation: compare different input sizes
!python evaluation/ablation/ablation_resolution.py --voc-root {VOC_ROOT}

## 10. Display All Plots & Download

Display every generated plot, then zip and download.

In [ ]:
import glob
from IPython.display import display, Image as IPImage

# Collect ALL generated png files
plot_dirs = [
    "results/compare",
    "results/metrics",
    "results/visualization",
    "results/infer_unet",
    "results/infer_deeplabv3plus",
    "results/infer_dinov2",
    "results/infer_sam2",
    "results/ablation",
]

all_plots = []
for d in plot_dirs:
    all_plots.extend(sorted(glob.glob(os.path.join(d, "**", "*.png"), recursive=True)))

print(f"Total plots found: {len(all_plots)}\n")

for fpath in all_plots:
    print(f"--- {fpath} ---")
    display(IPImage(filename=fpath, width=800))
    print()

In [ ]:
import shutil
from google.colab import files

# Zip all results (plots, metrics json, training logs, summary tables)
zip_path = "/content/261-mini2-results"
shutil.make_archive(zip_path, "zip", root_dir=REPO_DIR, base_dir="results")
print(f"Created: {zip_path}.zip")

# Trigger browser download
files.download(f"{zip_path}.zip")